

Once again consider four modeling options for house price:

    Using only the size and number of rooms.
    Using size, number of rooms, and building type.
    Using size and building type, and their interaction.
    Using a 5-degree polynomial on size, a 5-degree polynomial on number of rooms, and also building type.

Use cross_val_score with the pipelines you made earlier to find the cross-validated root mean squared error for each model.

Which do you prefer? Does this agree with your conclusion from earlier?


In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.compose import ColumnTransformer

In [2]:
ames = pd.read_csv("/content/AmesHousing(1).csv")

FileNotFoundError: [Errno 2] No such file or directory: '/content/AmesHousing(1).csv'

## Model 1 : Using only the size and number of rooms.

In [ ]:
X = ames.drop("SalePrice", axis = 1) # use everything in ames other than saleprice bcoz saleprice is y
y = ames["SalePrice"]

#X_train, X_test, y_train, y_test = train_test_split(X, y)

df_cross_mse = pd.DataFrame(columns =['Model', 'R^square'])


In [ ]:
ct = ColumnTransformer(
  [
    ("select", "passthrough", ["Gr Liv Area", "TotRms AbvGrd"])
  ],
  remainder = "drop"
)


lr_pipeline = Pipeline(
  [("preprocessing", ct),
  ("linear_regression", LinearRegression())]
)
lr_pipeline



scores = cross_val_score(lr_pipeline, X, y, cv=5, scoring='r2')
s1 = scores.mean()
df_cross_mse.loc[len(df_cross_mse)] = ['model 1', s1 ]

In [ ]:
scores.mean()

np.float64(0.504208752508862)

## Model 2 : Using size, number of rooms, and building type.

In [ ]:
ct_2 = ColumnTransformer(
  [
    ("dummify", OneHotEncoder(sparse_output = False), ["Bldg Type"]),
    ("select", "passthrough", ["Gr Liv Area", "TotRms AbvGrd"])
  ],
  remainder = "drop"
)

lr_pipeline_2 = Pipeline(
  [("preprocessing", ct_2),
  ("linear_regression", LinearRegression())]
)

lr_pipeline_2

scores_2 = cross_val_score(lr_pipeline_2, X, y, cv=5, scoring='r2')
print(scores_2)
s2 = scores_2.mean()
df_cross_mse.loc[len(df_cross_mse)] = ['model 2', s2 ]

[0.53197809 0.53225302 0.42829534 0.56574793 0.60613781]


## Model 3: Using size and building type, and their interaction.

In [ ]:
ct_3 = ColumnTransformer(
  [
    ("dummify", OneHotEncoder(sparse_output = False), ["Bldg Type"]),
    ("interaction", PolynomialFeatures(interaction_only = True), ["Gr Liv Area", "TotRms AbvGrd"])

  ],
  remainder = "drop"
)

lr_pipeline_3 = Pipeline(
  [("preprocessing", ct_3),
  ("linear_regression", LinearRegression())]
)

lr_pipeline_3

scores_3 = cross_val_score(lr_pipeline_3, X, y, cv=5, scoring='r2')
print(scores_3)
s3 = scores_3.mean()
df_cross_mse.loc[len(df_cross_mse)]= ['model 3', s3]

[0.52063325 0.53213069 0.44413214 0.58224516 0.59005814]


## Model 4 : Using a 5-degree polynomial on size, a 5-degree polynomial on number of rooms, and also building type.

In [ ]:
ct_4 = ColumnTransformer(
    [
    ('dummify', OneHotEncoder(sparse_output=False), ['Bldg Type']),
    ('polynomial', PolynomialFeatures(degree=5), ['Gr Liv Area', 'TotRms AbvGrd'])
    ]
)

pipeline_4 = Pipeline(
    [
        ('preprocessing', ct_4),
        ('linear regression', LinearRegression())
    ]
)

scores_4 = cross_val_score(pipeline_4, X,y, cv=5, scoring='r2')
print(scores_4)
s4=scores_4.mean()
df_cross_mse.loc[len(df_cross_mse)]= ['model 4', s4]
df_cross_mse

[ 0.49989506  0.49666289  0.08148884 -0.10265125  0.50965871]


,Model,R^square
0,model 1,0.504209
1,model 2,0.532882
2,model 3,0.533840
3,model 4,0.297011


Based on the camparison of the MSE of 4 differernt models, model 3 has the highest R square, making it the best model for MSE as critetia. Further exploration can be done to check if model 3 is the one to use int he real world. Although R^2 is a good metric to calculate the performance of a regresion model, it does not take into account the penalty for added variables. We can do further exploraion to find out if this is actually the best model or adding more variables just increased R^square without it being a better model.
 We don't want to misunderstand the data and the model, so it is always good to assume the worst case scenarios to filter out most possible errors the model. This is essentialy becuase in real world, the data is usually not clean and understaning the human meaning of it is necessary

Some definitions to demember:

**R-squared (R²)**, also known as the coefficient of determination, is a statistical measure that represents the proportion of the variance in the dependent variable that is predictable from the independent variables.

In simpler terms, R-squared tells you how well your model fits the data.

*   An R-squared of 1 means that the model explains all of the variability in the dependent variable.
*   An R-squared of 0 means that the model explains none of the variability in the dependent variable.
*   An R-squared between 0 and 1 indicates the proportion of variance explained by the model.

A higher R-squared generally indicates a better fit, but it's important to consider other factors and metrics as well.

**Mean Squared Error (MSE)** is a common metric used to evaluate the performance of regression models. It measures the average squared difference between the actual (observed) and predicted values.

A lower MSE indicates that the model's predictions are closer to the actual values, meaning the model is performing better.

# **Part 2**

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('dummify',
                                                                         OneHotEncoder(sparse_output=False),
                                                                         ['Bldg '
                                                                          'Type']),
                                                                        ('polynomial',
                                                                         PolynomialFeatures(),
                                                                         ['Gr '
                                                                          'Liv '
                                                                          'Area',
                                                                          'TotRms '
                                                                          'AbvGrd'])])),
                                       ('linear regression',
                                        LinearRegression())]),
             param_grid={'preprocessing__polynomial__degree': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10])},
             scoring='r2')

{'mean_fit_time': array([0.0481266 , 0.0990212 , 0.12396111, 0.03068514, 0.03425555,
        0.04802647, 0.04451547, 0.04391055, 0.02882972, 0.03763757]),
 'std_fit_time': array([0.03008816, 0.01319473, 0.03803271, 0.00494931, 0.01009276,
        0.02111938, 0.00335074, 0.01245739, 0.00083019, 0.00389184]),
 'mean_score_time': array([0.02359824, 0.04846935, 0.04809484, 0.0164443 , 0.01972175,
        0.02082338, 0.02824259, 0.01616955, 0.01352291, 0.01513095]),
 'std_score_time': array([0.0107252 , 0.01058697, 0.02173476, 0.00380892, 0.00551135,
        0.0051215 , 0.00911653, 0.00440994, 0.00042084, 0.00119518]),
 'param_preprocessing__polynomial__degree': masked_array(data=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
              mask=[False, False, False, False, False, False, False, False,
                    False, False],
        fill_value=999999),
 'params': [{'preprocessing__polynomial__degree': np.int64(1)},
  {'preprocessing__polynomial__degree': np.int64(2)},
  {'preprocessing__polynom

[0.048126602172851564,
 0.09902119636535645,
 0.12396111488342285,
 0.03068513870239258,
 0.03425555229187012,
 0.04802646636962891,
 0.044515466690063475,
 0.043910551071166995,
 0.0288297176361084,
 0.0376375675201416]